# 01. OpenAI API 기초
**SK하이닉스 Autonomous R&D — Day 3** 

> ChatGPT 웹과 **같은 모델**을 코드에서 호출.

---
## 0. 라이브러리 설치

아래 셀을 **최초 1회** 실행.

In [1]:
!pip install openai==1.58.1

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 19.3 MB/s  0:00:00

   -------- -------------------------------  3/15 [pydantic-core]
   ---------------- -----------------------  6/15 [h11]
   -------------------------- ------------- 10/15 [pydantic]
   -------------------------- ------------- 10/15 [pydantic]
   -------------------------- ------------- 10/15 [pydantic]
   ----------------------------- ---------- 11/15 [httpcore]
   -------------------------------- ------- 12/15 [anyio]
   ---------------------------------- ----- 13/15 [httpx]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   ------------------------------------- -- 14/15 [openai]
   -----------------

In [3]:
!pip install python-dotenv

---
## 1. LLM API란?

강의자료 요약:

| 구분 | ChatGPT 웹 | API |
|------|------------|-----|
| 사용자 | 사람이 직접 입력 | **프로그램**이 요청 |
| 결과 | 화면에 표시 | **response 객체**로 반환 |
| 본질 | 대화 | **Prompt → Text** |

API 요청의 핵심 3요소:
1. **`model`** — 어떤 LLM을 쓸지
2. **`messages`** — system / user (대화 내용)
3. **`temperature`** — 답변의 무작위성 (0=일관, 1=창의)

---
## 2. API 키 연결


처음에는 **키를 변수에 넣어** 바로 연결해 봅니다.

> OpenAI Platform → [API keys](https://platform.openai.com/api-keys) 에서 발급

In [ ]:
from openai import OpenAI

# 아래에 본인 키를 붙여넣고 실행
api_key = 'key'

client = OpenAI(api_key=api_key)

In [5]:
# 연결 테스트
test = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': 'Hi, reply with one word: OK'}],
    max_tokens=10,
)
print('연결 성공:', test.choices[0].message.content)

연결 성공: OK


1. **`.env` 파일**에 키 저장 (코드와 분리) (OPENAI_API_KEY=sk...)
2. **`.gitignore`**에 `.env` 등록 → Git에 **절대 올리지 않음**

In [43]:
from pathlib import Path

root = Path('..')  # ttest/
env_path = root / '.env' 
#env_path ='C:\Users\Admin\Desktop\실습폴더\.env' 이렇게 넣어도 상관 없음
gitignore_path = root / '.gitignore'

print('.env 존재:', env_path.exists(), '→', env_path.resolve())
print('.gitignore 존재:', gitignore_path.exists())

if gitignore_path.exists():
    content = gitignore_path.read_text(encoding='utf-8')
    print('.env in .gitignore:', '.env' in content)

.env 존재: True → C:\Users\Admin\Desktop\실습폴더\.env
.gitignore 존재: True
.env in .gitignore: True


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(root / '.env')
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    raise ValueError('ttest/.env에 OPENAI_API_KEY=sk-... 를 설정하세요.')

client = OpenAI(api_key=api_key)
print('✅ .env에서 API 키 로드 완료 (코드에 키 없음)')

ValueError: ttest/.env에 OPENAI_API_KEY=sk-... 를 설정하세요.

: 

---
## 3. 첫 API 호출 

`chat.completions.create()` — **messages** 리스트를 보내면 AI가 답변합니다.

| role | 역할 |
|------|------|
| `system` | AI 역할·규칙 (선택) |
| `user` | 사용자 질문 |

In [15]:
response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant.'},
        {'role': 'user',   'content': '2002년 월드컵 우승 팀이 어디야?'},
    ],
)

In [21]:
response.choices[0].message.content

'2002년 FIFA 월드컵 우승 팀은 브라질입니다. 브라질은 결승에서 독일을 2-0으로 이기고 5번째 월드컵 트로피를 차지했습니다.'

In [25]:
####### 2022년 월드컵 우승팀 물어보기
response = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.1,
    messages=[
        {'role': 'system', 'content':'You are a helpful assistant.'},
        {'role': 'user', 'content': '2022년 월드컵 우슨 팀이 어디야?'}
    ]
)


...

Ellipsis

In [26]:
response.choices[0].message.content

'2022년 FIFA 월드컵은 카타르에서 개최되었습니다. 이 대회는 2022년 11월 20일부터 12월 18일까지 진행되었습니다. 월드컵에서 우승한 팀은 아르헨티나로, 결승에서 프랑스를 상대로 승리하여 36년 만에 두 번째 월드컵 우승을 차지했습니다.'

---
## 4. 응답(Response) 객체

API는 **텍스트만**이 아니라 **객체**를 돌려줍니다.

| 필드 | 의미 |
|------|------|
| `response.choices[0].message.content` | **실제 답변 텍스트** ← 가장 많이 사용 |
| `response.choices[0].finish_reason` | `stop`=정상 종료, `length`=토큰 초과 |
| `response.usage.total_tokens` | 이번 호출에 쓴 토큰 수 (비용 참고) |
| `response.id` | 요청 ID (로그·디버깅용) |

In [30]:
print('ID:', response.id)
print('finish_reason:', response.choices[0].finish_reason)
print('답변:', response.choices[0].message.content)
print('토큰 사용:', response.usage.total_tokens, '(prompt:', response.usage.prompt_tokens,
      '/ completion:', response.usage.completion_tokens, ')')

ID: chatcmpl-DuBUjl1cNzB4JeMpfkcb50ynKoeuV
finish_reason: stop
답변: 2022년 FIFA 월드컵은 카타르에서 개최되었습니다. 이 대회는 2022년 11월 20일부터 12월 18일까지 진행되었습니다. 월드컵에서 우승한 팀은 아르헨티나로, 결승에서 프랑스를 상대로 승리하여 36년 만에 두 번째 월드컵 우승을 차지했습니다.
토큰 사용: 113 (prompt: 30 / completion: 83 )


---
## 5. System / User 프롬프트 

같은 `user` 질문이라도 **`system`에 역할·출력 규칙**을 주면 답변 **형식·깊이·관점**이 달라집니다.

아래는 **의도적으로 짧고 모호한 질문**을 사용합니다. system 유무에 따라 차이가 잘 보입니다.

| | system | user |
|---|--------|------|
| 역할 | 모델의 **역할·규칙·형식** | 이번 **질문·작업** |
| 비유 | 직무 기술서 | 오늘 업무 지시 |

In [31]:
# 질문을 짧고 모호하게 → system이 없으면 범용 답변, 있으면 역할·형식에 맞춘 답변
question = 'for문이 뭐야?'

r1 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[{'role': 'user', 'content': question}],
)

print('=== system 없음 (범용 설명, 길어질 수 있음) ===')
print(r1.choices[0].message.content)

=== system 없음 (범용 설명, 길어질 수 있음) ===
`for`문은 프로그래밍에서 반복문 중 하나로, 특정 조건을 만족하는 동안 코드 블록을 반복 실행하는 데 사용됩니다. 주로 리스트, 배열, 또는 다른 반복 가능한 객체의 요소를 순회할 때 많이 사용됩니다.

예를 들어, Python에서 `for`문을 사용하는 기본적인 예시는 다음과 같습니다:

```python
# 리스트의 각 요소를 출력하는 예제
fruits = ['사과', '바나나', '체리']
for fruit in fruits:
    print(fruit)
```

위의 코드에서는 `fruits` 리스트의 각 요소를 하나씩 꺼내어 `fruit` 변수에 저장하고, 그 값을 출력합니다. 이처럼 `for`문은 반복적인 작업을 간편하게 수행할 수 있게 해줍니다.

다른 프로그래밍 언어에서도 비슷한 구조를 가지고 있으며, 사용법은 약간씩 다를 수 있습니다.


In [32]:
r2 = client.chat.completions.create(
    model='gpt-4o-mini',
    temperature=0.2,
    messages=[
        {
            'role': 'system',
            'content': '''You are a Python tutor.
Rules:
- Answer in Korean only
- Use exactly 3 lines: [비유 1문장] / [코드 예시 max 3 lines] / [실무 한 줄]
- No long explanation, no bullet lists''',
        },
        {'role': 'user', 'content': question},
    ],
)

print('=== system 있음 (튜터 + 3줄 형식 강제) ===')
print(r2.choices[0].message.content)

=== system 있음 (튜터 + 3줄 형식 강제) ===
for문은 반복문으로, 마치 반복해서 노래를 부르는 것과 같아요.  
```python
for i in range(5):
    print(i)
```
리스트나 범위의 요소를 쉽게 순회할 수 있습니다.


---
## 6. temperature — **일반 예시**

같은 질문, 다른 `temperature` → 결과가 어떻게 달라지는지 확인합니다.

| temperature | 특징 | 용도 |
|-------------|------|------|
| 0 ~ 0.3 | 일관적, 사실 위주 | 분석, 판정, 코드 |
| 0.7 ~ 1.0 | 다양·창의적 | 브레인스토밍, 카피 |

In [33]:
question = '팀 워크숍 아이스브레이킹 아이디어 1가지만.'

for temp in [0.0, 0.7, 1.0]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=temp,
        messages=[{'role': 'user', 'content': question}],
    )
    print(f'[temperature={temp}] {r.choices[0].message.content}')
    print()

[temperature=0.0] "두 진실과 한 거짓" 게임을 제안합니다. 각 팀원이 자신에 대한 두 가지 진실과 하나의 거짓을 말합니다. 다른 팀원들은 어떤 것이 거짓인지 맞춰야 합니다. 이 게임은 서로에 대한 이해를 높이고, 자연스럽게 대화를 유도하는 데 도움이 됩니다. 재미있고 가벼운 분위기를 만들어 팀원 간의 친밀감을 증진시킬 수 있습니다.

[temperature=0.7] "두 진실과 한 거짓" 게임을 제안합니다. 각 참가자는 자신에 대한 두 가지 진실과 하나의 거짓을 이야기합니다. 다른 팀원들은 어떤 것이 거짓인지 추측해야 합니다. 이 게임은 서로에 대한 흥미로운 사실을 공유하고, 자연스럽게 대화를 이어갈 수 있는 좋은 방법입니다.

[temperature=1.0] 팀 워크숍에서 사용할 수 있는 아이스브레이킹 아이디어로 "두 가지 진실과 한 가지 거짓" 게임을 제안합니다.

**진행 방법:**
1. 팀원 각자가 자신에 대한 세 가지 사실을 준비합니다. 이 중 두 가지는 진실이고, 하나는 거짓입니다.
2. 차례를 돌아가며 각 팀원이 자신의 세 가지 사실을 발표합니다.
3. 다른 팀원들은 어떤 것이 거짓인지 맞춰봅니다.
4. 정답을 맞춘 후, 거짓인 사실에 대해 간단히 이야기하며 팀원들 간의 이해를 높입니다.

이 게임은 팀원들 간의 친밀감을 높이고, 서로에 대한 새로운 정보를 공유하는 좋은 기회가 됩니다.



---
## 7. `max_tokens` — 출력 길이 제한

답변이 너무 길어지는 것을 막거나, 비용을 줄일 때 사용합니다.

In [34]:
for max_tok in [20, 100]:
    r = client.chat.completions.create(
        model='gpt-4o-mini',
        temperature=0.3,
        max_tokens=max_tok,
        messages=[{'role': 'user', 'content': '대한민국의 역사를 요약해줘.'}],
    )
    print(f'--- max_tokens={max_tok} (finish: {r.choices[0].finish_reason}) ---')
    print(r.choices[0].message.content)
    print()

--- max_tokens=20 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 �

--- max_tokens=100 (finish: length) ---
대한민국의 역사는 고대부터 현대에 이르기까지 다양한 사건과 변화를 겪어왔습니다. 다음은 대한민국 역사의 주요 사건과 흐름을 요약한 것입니다.

1. **고대 및 삼국 시대 (기원전 2333년 ~ 668년)**:
   - 고조선(기원전 2333년 설립)으로 시작하여, 후에 백제, 신라, 고구려의 삼국이 형



---
## 8. `ask()` 함수 — 재사용 패턴

같은 API 호출을 함수로 감싸기.

In [35]:
def ask(user_message, system_message='You are a helpful assistant.', temperature=0.2, max_tokens=None):
    """1턴 질의 — 답변 텍스트만 반환"""
    kwargs = dict(
        model='gpt-4o-mini',
        temperature=temperature,
        messages=[
            {'role': 'system', 'content': system_message},
            {'role': 'user',   'content': user_message},
        ],
    )
    if max_tokens is not None:
        kwargs['max_tokens'] = max_tokens
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message.content


print(ask('서울 3일 여행 코스 추천해줘. bullet 3개만.', max_tokens=150))

서울 3일 여행 코스 추천드립니다:

- **1일차: 역사와 문화 탐방**
  - 경복궁 방문: 고궁 탐방과 국립민속박물관 관람
  - 북촌 한옥마을 산책: 전통 한옥과 예쁜 카페 탐방
  - 인사동 거리: 전통 공예품 쇼핑 및 찻집 체험

- **2일차: 현대 서울과 쇼핑**
  - 명동: 쇼핑과 길거리 음식 즐기기
  - 남산타워: 서울 전경 감상 및 케이블카 체험
  - 동대문 디자인 플라자(DDP): 현대 건


---
## 9. 멀티턴 대화 

이전 대화를 `messages`에 **그대로 이어 붙이면** 후속 질문이 가능합니다.

```
system → user → assistant → user → ...
```

### 기존대화

In [40]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)

messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '2일차만 더 자세히. 맛집 2곳 포함.'},
]

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

=== 1턴 ===
제주도 2박 3일 여행 계획입니다.

**1일차:** 제주공항 도착 후 한라산 등반 및 성산일출봉 관람, 저녁에는 제주 해산물로 저녁식사.

**2일차:** 협재 해수욕장에서 해수욕 및 수영, 오후에는 오름 탐방 후 제주 전통 시장에서 쇼핑.

**3일차:** 만장굴 탐방 후 우도 방문, 마지막으로 제주도 특산물 구매 후 제주공항으로 출발.

=== 2턴 (맥락 유지) ===
2일차 여행 일정을 자세히 안내해드리겠습니다.

### 2일차 일정

**아침:**
- **조식**: 호텔에서 제공하는 조식을 즐기세요. 다양한 메뉴가 있을 것입니다.

**오전:**
- **관광지 1: 경복궁**
  - 경복궁은 조선 왕조의 첫 번째 궁궐로, 아름다운 건축물과 정원이 매력적입니다. 궁궐 내부를 탐방하고, 수문장 교대식도 관람해보세요.

**점심:**
- **맛집 1: 광화문 국밥**
  - 위치: 경복궁 근처
  - 추천 메뉴: 소고기 국밥
  - 설명: 따뜻한 국밥 한 그릇으로 배를 채우고, 전통적인 한국의 맛을 느껴보세요.

**오후:**
- **관광지 2: 북촌 한옥마을**
  - 전통 한옥이 잘 보존된 지역으로, 한국의 전통 문화를 체험할 수 있습니다. 골목길을 따라 산책하며 사진도 찍어보세요.

**저녁:**
- **맛집 2: 삼청동 수제비**
  - 위치: 북촌 한옥마을 근처
  - 추천 메뉴: 해물 수제비
  - 설명: 쫄깃한 수제비와 신선한 해물로 만든 국물이 일품입니다. 따뜻한 저녁식사를 즐기기에 좋은 곳입니다.

**저녁 후:**
- **산책**: 삼청동 거리에서 여유롭게 산책하며 카페나 작은 상점들을 둘러보세요.

이렇게 2일차 일정을 구성해보았습니다. 즐거운 여행 되세요!


### 멀티턴

### Assistant 메시지는 앞서 있었던 prompt들(user/system)에 대한 모델의 응답을 구성하며, 종종 대화 상태를 유지하기 위해 히스토리의 일부로 저장됩니다.

#### 특징
모델이 생성한 응답을 포함한다.

이전 message의 context 처리를 위해 대화의 연속성 유지를 돕는다.

다음 API 호출에서는 assistant 메시지를 포함시켜야 문맥이 이어짐.

In [41]:
messages = [
    {'role': 'system', 'content': 'You are a helpful travel assistant. Answer in Korean.'},
    {'role': 'user',   'content': '제주도 2박 3일 여행 계획 짜줘. 하루 요약 1줄씩.'},
]

r1 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
answer1 = r1.choices[0].message.content
print('=== 1턴 ===')
print(answer1)

messages.append({'role': 'assistant', 'content': answer1})
messages.append({'role': 'user', 'content': '2일차만 더 자세히. 맛집 2곳 포함.'})

r2 = client.chat.completions.create(model='gpt-4o-mini', temperature=0.3, messages=messages)
print('\n=== 2턴 (맥락 유지) ===')
print(r2.choices[0].message.content)

=== 1턴 ===
제주도 2박 3일 여행 계획을 아래와 같이 짜보았습니다.

**1일차:** 제주공항 도착 후 한라산 국립공원에서 하이킹을 즐기고, 저녁에는 제주시의 유명한 흑돼지 BBQ를 맛본다.

**2일차:** 성산일출봉에서 일출을 감상한 후, 우도 섬으로 배를 타고 가서 자전거 투어와 해변에서 여유로운 시간을 보낸다.

**3일차:** 만장굴과 협재해수욕장을 방문한 후, 제주도 특산물 쇼핑을 즐기고 제주공항으로 돌아간다.

=== 2턴 (맥락 유지) ===
2일차의 자세한 계획은 다음과 같습니다.

**아침:** 성산일출봉에서 일출을 감상한 후, 근처의 **성산일출봉 해물뚝배기**에서 신선한 해물뚝배기로 아침 식사를 즐깁니다.

**오전:** 성산일출봉을 하산한 후, 우도행 배를 타고 우도로 이동합니다. 우도에 도착하면 자전거를 대여하여 섬을 둘러보며 아름다운 경치를 감상합니다.

**점심:** 우도에서 유명한 **땅콩 아이스크림**과 함께 우도의 특산물인 땅콩을 활용한 음식을 맛보세요. 또한, **우도 해수욕장** 근처의 **우도 뚝배기**에서 신선한 해산물 요리를 즐길 수 있습니다.

**오후:** 우도의 해변에서 여유롭게 시간을 보내고, 해수욕이나 스노클링을 즐깁니다. 이후 배를 타고 제주 본섬으로 돌아옵니다.

**저녁:** 제주 본섬으로 돌아온 후, **제주 흑돼지 거리**에 위치한 **돈사돈**에서 제주 흑돼지 BBQ를 맛보며 하루를 마무리합니다. 

이렇게 2일차를 알차게 보내실 수 있습니다!


---
## ✏️ 실습 1

아래 주제로 **2턴** 대화를 만들어 보세요.

1턴: "Python for문 기본 문법 설명해줘"
2턴: "방금 설명한 걸로 1~5 합 구하는 코드 예시만 보여줘"

In [ ]:
# ── 여기에 작성 ──
messages = [
    {'role': 'system', 'content': 'You are a Python tutor. Answer in Korean.'},
    # {'role': 'user', 'content': '...'},
]
# r1 = client.chat.completions.create(...)
# messages.append(...)
# r2 = client.chat.completions.create(...)

---
## ✏️ 실습 1

커서 프롬프트 : 3일차 폴더에 스트림릿과 실습코드의 openai api를 이용해서 간단한 챗봇을 만드는 python 코드 만들어줘

In [42]:
!pip install streamlit

  Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl.metadata (59 kB)
   ---------------------------------------- 0.0/10.1 MB ? eta -:--:--
   ----------- ---------------------------- 2.9/10.1 MB 14.0 MB/s eta 0:00:01
   --------------------------- ------------ 6.8/10.1 MB 17.5 MB/s eta 0:00:01
   -------------------------------------- - 9.7/10.1 MB 15.5 MB/s eta 0:00:01
   ---------------------------------------- 10.1/10.1 MB 13.4 MB/s  0:00:00
   ---------------------------------------- 0.0/731.2 kB ? eta -:--:--
   ---------------------------------------- 731.2/731.2 kB 10.0 MB/s  0:00:00
Using cached numpy-2.0.2-cp39-cp39-win_amd64.whl (15.9 MB)
   ---------------------------------------- 0.0/11.4 MB ? eta -:--:--
   --------- ------------------------------ 2.6/11.4 MB 16.7 MB/s eta 0:00:01
   --------------- ------------------------ 4.5/11.4 MB 11.2 MB/s eta 0:00:01
   -------------------- ------------------- 5.8/11.4 MB 9.3 MB/s eta 0:00:01
   --------------------------------- ----